In [1]:
import pandas as pd
from pathlib import Path

base = Path("/kaggle/input/notebooks/syedjaffarrazakazmi")
oof_paths = {
    "config1_no_augmentation": base / "03-baseline-pipeline/results/oof_predictions.csv",
    "config2_geometric_augmentation": base / "04-config2-geometric-augmentation/results/oof_predictions.csv",
    "config3_photometric_augmentation": base / "05-config3-photometric-augmentation/results/oof_predictions.csv",
    "config4_combined_augmentation": base / "06-config4-combined-augmentation/results/oof_predictions.csv",
    "config5_mixup": base / "07-config6-mixup/results/oof_predictions.csv",
    "config6_cutmix": base / "08-config6-cutmix/results/oof_predictions.csv",
    "config7_randaugment": base / "09-config7-rand-augmentation/results/oof_predictions.csv",
}

for name, p in oof_paths.items():
    print(f"{name}: {'FOUND' if p.exists() else 'MISSING'}")

first = pd.read_csv(oof_paths["config1_no_augmentation"])
print("\nColumns:", first.columns.tolist())
print(first.head(3))

config1_no_augmentation: FOUND
config2_geometric_augmentation: FOUND
config3_photometric_augmentation: FOUND
config4_combined_augmentation: FOUND
config5_mixup: FOUND
config6_cutmix: FOUND
config7_randaugment: FOUND

Columns: ['file_name', 'patient_id', 'fold', 'label', 'label_name', 'pred', 'pred_name', 'prob_Healthy', 'prob_Benign', 'prob_OPMD', 'prob_OCA']
     file_name patient_id  fold  label label_name  pred pred_name  \
0  R-01-01.jpg       R-01     3      2       OPMD     2      OPMD   
1  R-01-02.jpg       R-01     3      2       OPMD     2      OPMD   
2  R-01-03.jpg       R-01     3      1     Benign     0   Healthy   

   prob_Healthy  prob_Benign  prob_OPMD  prob_OCA  
0      0.003729     0.009032   0.986987  0.000252  
1      0.000415     0.009285   0.984925  0.005374  
2      0.999966     0.000012   0.000019  0.000002  


In [2]:
import pandas as pd

wrong_counts = {}   # file_name -> number of configs that got it wrong
patient_of = {}     # file_name -> patient_id

for cfg_name, path in oof_paths.items():
    oof = pd.read_csv(path)
    oca = oof[oof["label_name"] == "OCA"]
    for _, row in oca.iterrows():
        patient_of[row["file_name"]] = row["patient_id"]
        if row["pred_name"] != "OCA":
            wrong_counts[row["file_name"]] = wrong_counts.get(row["file_name"], 0) + 1

# Include OCA images that were never wrong, so totals add up to 129
all_oca = pd.DataFrame({"file_name": list(patient_of.keys())})
all_oca["patient_id"] = all_oca["file_name"].map(patient_of)
all_oca["times_wrong"] = all_oca["file_name"].map(wrong_counts).fillna(0).astype(int)

print(f"Total OCA images: {len(all_oca)}")
print(f"Wrong in all 7 configs: {(all_oca['times_wrong'] == 7).sum()}")
print(f"Wrong in 5+ configs:    {(all_oca['times_wrong'] >= 5).sum()}")
print(f"Never wrong:            {(all_oca['times_wrong'] == 0).sum()}")

print("\nDistribution of times_wrong (how many images were wrong in N configs):")
print(all_oca["times_wrong"].value_counts().sort_index())

# Patient-level view
by_patient = (all_oca.groupby("patient_id")
              .agg(n_images=("file_name", "count"), avg_times_wrong=("times_wrong", "mean"))
              .sort_values("avg_times_wrong", ascending=False))
print(f"\nOCA patients: {len(by_patient)}")
print("\nWorst patients (by avg times wrong):")
print(by_patient.head(10))

Total OCA images: 129
Wrong in all 7 configs: 46
Wrong in 5+ configs:    82
Never wrong:            8

Distribution of times_wrong (how many images were wrong in N configs):
times_wrong
0     8
1     7
2    17
3     6
4     9
5    12
6    24
7    46
Name: count, dtype: int64

OCA patients: 61

Worst patients (by avg times wrong):
            n_images  avg_times_wrong
patient_id                           
K-009              1              7.0
N-342              2              7.0
N-302              5              7.0
N-020              1              7.0
R-05               2              7.0
R-136              1              7.0
R-145              1              7.0
R-10               1              7.0
S-87               1              7.0
S-132              1              7.0
